# GBL Score & Win Predictor

This notebook is a thin driver on top of the modules in [src/](src/). The
actual logic lives there — edit those files, not this notebook.

## Daily workflow

1. **New results came in?** Append rows to [data/raw/gbl_all_teams_full.csv](data/raw/gbl_all_teams_full.csv) — only the first 5 columns matter (`game_date, home_team, away_team, home_score, away_score`).
2. **New fixtures to predict?** Edit [data/raw/upcoming_fixtures.csv](data/raw/upcoming_fixtures.csv) — one row per game with columns `fixture_date, matchday, home_team, away_team`.
3. **Retrain** — re-run cell 2 (or `python scripts/train.py`).
4. **Predict** — re-run cell 3.

Team names in step 2 must **exactly** match names used in step 1 — the pipeline now validates this and will raise with a `did you mean …?` suggestion if there's a typo.

If you change anything in `src/`, restart the kernel — Python caches modules after the first import.

In [7]:
# Install from requirements.txt — everything the project needs, pinned in one place.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Train the model end-to-end and print the honest (walk-forward) metrics.
# This rebuilds features from the raw CSV, runs a time-based train/test split,
# then refits on all available games and saves the artifact to models/.
from src.model import _format_metrics, train_and_save

bundle = train_and_save()
print(_format_metrics(bundle.metrics))

  Train / test games: 123 / 31

  Win classifier:
    Accuracy     : 0.710
    Log loss     : 0.837  (lower is better)
    Brier score  : 0.239  (lower is better)

  Margin regressor (home_score - away_score):
    MAE          : 11.84 pts
    RMSE         : 14.73 pts
    R^2          : 0.088

  Total regressor (home_score + away_score):
    MAE          : 14.62 pts
    RMSE         : 17.91 pts
    R^2          : 0.132

  Elo baseline (for comparison):
    Accuracy     : 0.742
    Log loss     : 0.568
    Brier score  : 0.191

  Ensemble (ML * 0.45 + Elo * 0.55):
    Accuracy     : 0.710
    Log loss     : 0.547
    Brier score  : 0.186



In [ ]:
# Predicted vs actual — the walk-forward backtest table.
#
# Every row below is a game the model did NOT see during training.
# So "correct = True" means the model genuinely got it right, not memorized.
from IPython.display import display

from src.backtest import build_comparison_table, comparison_summary

print(comparison_summary(bundle))
print()
display(build_comparison_table(bundle))

In [ ]:
# Upcoming fixtures + on-demand predictions. All the widget wiring lives
# in src/ui.py so this cell stays readable.
from src.ui import show_predictions_ui

show_predictions_ui(bundle)